# Interactive Cycle Time Analysis

This notebook provides an interactive interface for analyzing JIRA cycle times with clickable scatter plots that link directly to JIRA issues.

## Features
- Interactive credential input for JIRA connection
- Custom JQL query input
- Clickable scatter plot with JIRA issue links
- Hover tooltips with issue details
- Configurable workflow mapping

In [ ]:
# Import required libraries
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import getpass
import datetime
from jira import JIRA
import logging

# Import project modules
from jira_agile_metrics.querymanager import QueryManager
from jira_agile_metrics.calculators.cycletime import calculate_cycle_times
from jira_agile_metrics.calculators.scatterplot import calculate_scatterplot_data

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✅ Libraries imported successfully")

## 1. JIRA Connection Setup

Enter your JIRA credentials and server information:

In [ ]:
# JIRA Connection Configuration
print("🔐 JIRA Connection Setup")
print("=" * 50)

# Create input widgets
server_input = widgets.Text(
    value='https://your-company.atlassian.net',
    placeholder='Enter JIRA server URL',
    description='JIRA Server:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

username_input = widgets.Text(
    value='',
    placeholder='Enter your email/username',
    description='Username:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

password_input = widgets.Password(
    value='',
    placeholder='Enter API token or password',
    description='API Token:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

connect_button = widgets.Button(
    description='Connect to JIRA',
    button_style='primary',
    layout=widgets.Layout(width='200px')
)

connection_status = widgets.HTML(value="")

# Display widgets
display(server_input, username_input, password_input, connect_button, connection_status)

# Global variables
jira_client = None
query_manager = None

def connect_to_jira(b):
    global jira_client, query_manager
    
    try:
        connection_status.value = "🔄 Connecting to JIRA..."
        
        # Create JIRA connection
        jira_client = JIRA(
            server=server_input.value,
            basic_auth=(username_input.value, password_input.value)
        )
        
        # Test connection
        user = jira_client.current_user()
        
        # Create basic settings for QueryManager
        settings = {
            'max_results': None,
            'attributes': {},
            'known_values': {}
        }
        
        query_manager = QueryManager(jira_client, settings)
        
        connection_status.value = f"✅ Connected successfully as {user}"
        
    except Exception as e:
        connection_status.value = f"❌ Connection failed: {str(e)}"
        jira_client = None
        query_manager = None

connect_button.on_click(connect_to_jira)

## 2. Workflow Configuration

Define your workflow stages and their corresponding JIRA statuses:

In [ ]:
# Default workflow configuration
default_cycle = [
    {"name": "Backlog", "statuses": ["Backlog", "New", "Open"]},
    {"name": "Committed", "statuses": ["To Do", "Ready", "Next"]},
    {"name": "In Progress", "statuses": ["In Progress", "Development"]},
    {"name": "Review", "statuses": ["Code Review", "Review", "Testing"]},
    {"name": "Done", "statuses": ["Done", "Closed", "Resolved"]}
]

# Workflow configuration widget
workflow_text = widgets.Textarea(
    value=str(default_cycle),
    placeholder='Enter workflow configuration as Python list',
    description='Workflow:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='800px', height='150px')
)

committed_column_input = widgets.Text(
    value='Committed',
    description='Committed Stage:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

done_column_input = widgets.Text(
    value='Done',
    description='Done Stage:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

print("⚙️ Workflow Configuration")
print("=" * 50)
print("Configure your workflow stages and the statuses that map to each stage.")
print("Cycle time will be calculated from Committed Stage to Done Stage.")
print()

display(workflow_text)
display(widgets.HBox([committed_column_input, done_column_input]))

## 3. JQL Query Input

Enter your JQL query to fetch the issues you want to analyze:

In [ ]:
# JQL Query Configuration
jql_input = widgets.Textarea(
    value='project = "YOUR_PROJECT" AND issuetype = Story AND status in (Done, Closed) ORDER BY resolved DESC',
    placeholder='Enter your JQL query',
    description='JQL Query:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='800px', height='100px')
)

max_results_input = widgets.IntText(
    value=100,
    description='Max Results:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px')
)

analyze_button = widgets.Button(
    description='Analyze Issues',
    button_style='success',
    layout=widgets.Layout(width='200px')
)

analysis_status = widgets.HTML(value="")

print("📊 JQL Query Configuration")
print("=" * 50)
print("Enter a JQL query to fetch the issues you want to analyze.")
print("Example: project = 'MYPROJ' AND issuetype = Story AND resolved >= -90d")
print()

display(jql_input)
display(max_results_input)
display(analyze_button)
display(analysis_status)

# Global variables for analysis results
cycle_data = None
scatter_data = None

def analyze_issues(b):
    global cycle_data, scatter_data
    
    if not jira_client:
        analysis_status.value = "❌ Please connect to JIRA first"
        return
    
    try:
        analysis_status.value = "🔄 Fetching and analyzing issues..."
        
        # Parse workflow configuration
        cycle = eval(workflow_text.value)
        
        # Update settings
        settings = {
            'cycle': cycle,
            'attributes': {},
            'committed_column': committed_column_input.value,
            'done_column': done_column_input.value,
            'queries': [{'jql': jql_input.value, 'value': 'Analysis'}],
            'query_attribute': None,
            'max_results': max_results_input.value
        }
        
        # Update query manager settings
        query_manager.settings.update(settings)
        
        # Calculate cycle times
        cycle_data = calculate_cycle_times(
            query_manager,
            cycle,
            {},  # attributes
            committed_column_input.value,
            done_column_input.value,
            [{'jql': jql_input.value, 'value': 'Analysis'}],
            None  # query_attribute
        )
        
        # Calculate scatter plot data
        scatter_data = calculate_scatterplot_data(cycle_data)
        
        analysis_status.value = f"✅ Analysis complete! Found {len(cycle_data)} issues, {len(scatter_data)} with cycle times"
        
    except Exception as e:
        analysis_status.value = f"❌ Analysis failed: {str(e)}"
        cycle_data = None
        scatter_data = None

analyze_button.on_click(analyze_issues)

## 4. Interactive Cycle Time Scatter Plot

Generate an interactive scatter plot with clickable data points:

In [ ]:
def create_interactive_scatter_plot():
    if scatter_data is None or len(scatter_data) == 0:
        print("❌ No data available. Please run the analysis first.")
        return
    
    # Prepare data for plotting
    plot_data = scatter_data.copy()
    
    # Convert cycle_time to days (numeric)
    plot_data['cycle_time_days'] = plot_data['cycle_time'].dt.total_seconds() / (24 * 3600)
    
    # Create hover text with issue details
    plot_data['hover_text'] = (
        "<b>" + plot_data['key'] + "</b><br>" +
        "Summary: " + plot_data['summary'].str[:50] + "...<br>" +
        "Cycle Time: " + plot_data['cycle_time_days'].round(1).astype(str) + " days<br>" +
        "Completed: " + plot_data['completed_date'].dt.strftime('%Y-%m-%d') + "<br>" +
        "Status: " + plot_data['status'] + "<br>" +
        "Type: " + plot_data['issue_type'] + "<br>" +
        "Blocked Days: " + plot_data['blocked_days'].astype(str) + "<br>" +
        "<i>Click to open in JIRA</i>"
    )
    
    # Create the scatter plot
    fig = go.Figure()
    
    # Add scatter trace
    fig.add_trace(go.Scatter(
        x=plot_data['completed_date'],
        y=plot_data['cycle_time_days'],
        mode='markers',
        marker=dict(
            size=8,
            color=plot_data['blocked_days'],
            colorscale='Reds',
            colorbar=dict(title="Blocked Days"),
            line=dict(width=1, color='DarkSlateGrey')
        ),
        text=plot_data['hover_text'],
        hovertemplate='%{text}<extra></extra>',
        customdata=plot_data['url'],
        name='Issues'
    ))
    
    # Add percentile lines
    percentiles = [0.5, 0.85, 0.95]
    colors = ['green', 'orange', 'red']
    
    for i, p in enumerate(percentiles):
        percentile_value = plot_data['cycle_time_days'].quantile(p)
        fig.add_hline(
            y=percentile_value,
            line_dash="dash",
            line_color=colors[i],
            annotation_text=f"{int(p*100)}% ({percentile_value:.1f} days)",
            annotation_position="top left"
        )
    
    # Update layout
    fig.update_layout(
        title={
            'text': 'Interactive Cycle Time Scatter Plot<br><sub>Click on data points to open JIRA issues</sub>',
            'x': 0.5,
            'xanchor': 'center'
        },
        xaxis_title='Completion Date',
        yaxis_title='Cycle Time (Days)',
        hovermode='closest',
        height=600,
        showlegend=False
    )
    
    # Add JavaScript for click handling
    fig.update_layout(
        updatemenus=[
            dict(
                type="buttons",
                direction="left",
                buttons=list([
                    dict(
                        args=[{"visible": [True]}],
                        label="Refresh",
                        method="restyle"
                    )
                ]),
                pad={"r": 10, "t": 10},
                showactive=True,
                x=0.01,
                xanchor="left",
                y=1.02,
                yanchor="top"
            ),
        ]
    )
    
    # Display the plot
    fig.show()
    
    # Add JavaScript for click handling
    display(HTML("""
    <script>
    // Add click handler for opening JIRA links
    document.addEventListener('DOMContentLoaded', function() {
        // Find all plotly graphs
        var graphs = document.querySelectorAll('.plotly-graph-div');
        
        graphs.forEach(function(graph) {
            graph.on('plotly_click', function(data) {
                if (data.points && data.points.length > 0) {
                    var point = data.points[0];
                    if (point.customdata) {
                        window.open(point.customdata, '_blank');
                    }
                }
            });
        });
    });
    </script>
    """))
    
    # Display summary statistics
    print("\n📈 Summary Statistics")
    print("=" * 50)
    print(f"Total Issues: {len(plot_data)}")
    print(f"Average Cycle Time: {plot_data['cycle_time_days'].mean():.1f} days")
    print(f"Median Cycle Time: {plot_data['cycle_time_days'].median():.1f} days")
    print(f"85th Percentile: {plot_data['cycle_time_days'].quantile(0.85):.1f} days")
    print(f"95th Percentile: {plot_data['cycle_time_days'].quantile(0.95):.1f} days")
    print(f"Average Blocked Days: {plot_data['blocked_days'].mean():.1f} days")

# Button to generate the plot
plot_button = widgets.Button(
    description='Generate Interactive Plot',
    button_style='info',
    layout=widgets.Layout(width='250px')
)

def generate_plot(b):
    create_interactive_scatter_plot()

plot_button.on_click(generate_plot)

print("📊 Interactive Visualization")
print("=" * 50)
print("Click the button below to generate an interactive scatter plot.")
print("Data points are colored by blocked days and clickable to open JIRA issues.")
print()

display(plot_button)

## 5. Data Export

Export your analysis results for further processing:

In [ ]:
def export_data():
    if cycle_data is None:
        print("❌ No data available. Please run the analysis first.")
        return
    
    # Export cycle data
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Full cycle data
    cycle_filename = f"cycle_time_data_{timestamp}.csv"
    cycle_data.to_csv(cycle_filename, index=False)
    print(f"✅ Full cycle data exported to: {cycle_filename}")
    
    # Scatter plot data
    if scatter_data is not None:
        scatter_filename = f"scatter_plot_data_{timestamp}.csv"
        scatter_data.to_csv(scatter_filename, index=False)
        print(f"✅ Scatter plot data exported to: {scatter_filename}")
    
    # Summary statistics
    if scatter_data is not None and len(scatter_data) > 0:
        cycle_time_days = scatter_data['cycle_time'].dt.total_seconds() / (24 * 3600)
        summary_stats = {
            'Total Issues': len(scatter_data),
            'Average Cycle Time (days)': cycle_time_days.mean(),
            'Median Cycle Time (days)': cycle_time_days.median(),
            '85th Percentile (days)': cycle_time_days.quantile(0.85),
            '95th Percentile (days)': cycle_time_days.quantile(0.95),
            'Average Blocked Days': scatter_data['blocked_days'].mean()
        }
        
        summary_df = pd.DataFrame(list(summary_stats.items()), columns=['Metric', 'Value'])
        summary_filename = f"summary_statistics_{timestamp}.csv"
        summary_df.to_csv(summary_filename, index=False)
        print(f"✅ Summary statistics exported to: {summary_filename}")

export_button = widgets.Button(
    description='Export Data',
    button_style='warning',
    layout=widgets.Layout(width='150px')
)

export_button.on_click(lambda b: export_data())

print("💾 Data Export")
print("=" * 50)
print("Export your analysis results to CSV files for further processing.")
print()

display(export_button)

## Usage Instructions

1. **Connect to JIRA**: Enter your JIRA server URL, username, and API token
2. **Configure Workflow**: Adjust the workflow stages to match your JIRA setup
3. **Enter JQL Query**: Specify which issues to analyze
4. **Run Analysis**: Click "Analyze Issues" to fetch and process data
5. **Generate Plot**: Create the interactive scatter plot
6. **Interact**: Click on data points to open JIRA issues in new tabs
7. **Export**: Save results to CSV files for further analysis

## Features

- **Interactive Scatter Plot**: Hover for details, click to open JIRA issues
- **Color Coding**: Points colored by blocked days (red = more blocked time)
- **Percentile Lines**: 50th, 85th, and 95th percentile indicators
- **Summary Statistics**: Key metrics displayed below the plot
- **Data Export**: Export raw data and statistics to CSV

## Tips

- Use API tokens instead of passwords for better security
- Start with a small dataset (limit results) to test your configuration
- Adjust workflow stages to match your team's process
- Use JQL filters to focus on specific time periods or issue types